# Notebook 13 — Capstone: Script / Reading-Order Classifier

This capstone addresses a **real production use case**: given an image that may contain text,
decide whether the text is **Left-to-Right (LTR)** — i.e., pure English — or **Right-to-Left (RTL)**
— i.e., pure Arabic, or a mix of English and Arabic (which is read RTL because of the Arabic
content).

A downstream OCR or rendering pipeline needs to know the reading direction **before** it processes
the image. Running a tiny image classifier first — a pure CNN forward pass, tens of milliseconds —
is orders of magnitude faster than running OCR on every image just to detect the script.

## Problem framing

- **Label 0 — LTR**: English-only images. Signs, screenshots, web pages, book pages.
- **Label 1 — RTL**: Arabic-only **or** mixed Arabic+English images. Any presence of Arabic flips the
  dominant reading order to RTL for downstream processing.

Note: we deliberately treat *mixed* as RTL. In UAE / Saudi signage, shop fronts, or bilingual
documents, the Arabic content drives the canonical reading order in practice.

## Learning goals

1. Design a **data-collection strategy** for a real, small, unusual binary problem.
2. Auto-label using a Unicode-range heuristic for bootstrapping.
3. Handle a dataset with a **directory-based** layout (`ImageFolder`).
4. Build a pipeline with a critical domain-specific augmentation caveat (no horizontal flip).
5. Two-stage training: head-only warm-up, then discriminative-LR full fine-tune.
6. Binary-classification evaluation: precision/recall/F1, ROC, PR, threshold tuning for a
   recall-for-minority-class target.
7. **Grad-CAM as a debug tool**: verify the model attends to text, not to background.
8. **ONNX export** and a portable one-shot inference function.
9. Honest comparison to non-ML alternatives (OCR + Unicode, text-density heuristic).


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/13_capstone_script_classifier.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Why not just run OCR + a Unicode check?

This is the first question an experienced engineer will ask, and the honest answer is "sometimes
you should!". Let's lay out the trade-offs up front so the reader knows why we're building a
classifier rather than reaching for `pytesseract`:

| Approach | Pros | Cons |
|---|---|---|
| **OCR + Unicode range check** | Deterministic. No training data. Handles arbitrary scripts. | Slow (100s of ms per image). Fragile on stylised fonts, low resolution, noisy backgrounds. Requires good preprocessing. |
| **Image classifier (this notebook)** | ~10 ms/image on GPU, ~30 ms CPU. Robust to font style. Mobile-deployable. | Needs labelled data. Only answers a fixed label set. Can be fooled by domain shift. |
| **Simple text-density heuristic** | Zero training. Zero ML dependencies. Great as a guard-rail. | Fails on images with mixed content or text scattered across the image. |

We'll build the classifier **and** show the alternatives so you know which tool to pull for which job.


## 2. Data strategy

For a custom binary classifier you will need to collect ~500–5000 images per class. Here is a
pragmatic sourcing plan:

**Class 0 — LTR (English-only)**

- English-language book-page scans.
- English-language street signs, shop fronts, menus.
- English text screenshots: news sites, Wikipedia articles, slide decks.
- Receipts, invoices, product labels in English.

**Class 1 — RTL (Arabic-only OR mixed)**

- Arabic-language book or newspaper scans.
- Arabic signage (standalone).
- UAE / Saudi / Egypt bilingual signage (Arabic on top + English below, or vice versa).
- Product packaging showing both scripts.
- Bilingual documents (government forms, menus).

Aim for **font, background, and lighting diversity** in *each* class. If every Arabic photo is a
neon-lit night-time shot and every English photo is a daytime indoor screenshot, the model will
learn "bright vs dark" rather than "script".

**Splits**: hold out 15% for validation and another 15% for test. Make sure test splits have
images *from different source batches* than train — otherwise you're effectively leaking.


## 3. Auto-labelling with a Unicode-range heuristic

If you happen to have the **source text** for each image (e.g., you rendered the image from known
strings, or you ran OCR once during collection), you can auto-label cheaply:

In [ ]:
def has_arabic(text: str) -> bool:
    """Return True if `text` contains any Arabic-script character.

    Covers the three main Arabic Unicode blocks:
      U+0600..U+06FF  Arabic
      U+0750..U+077F  Arabic Supplement
      U+08A0..U+08FF  Arabic Extended-A
    """
    for ch in text:
        cp = ord(ch)
        if 0x0600 <= cp <= 0x06FF:
            return True
        if 0x0750 <= cp <= 0x077F:
            return True
        if 0x08A0 <= cp <= 0x08FF:
            return True
    return False

def auto_label(source_text: str) -> int:
    # 0 = LTR, 1 = RTL (any Arabic present).
    return 1 if has_arabic(source_text) else 0

# quick sanity check
assert auto_label('Hello world') == 0
assert auto_label('مرحبا بالعالم') == 1
assert auto_label('Welcome مرحبا') == 1   # mixed -> RTL
assert auto_label('') == 0
print('auto_label OK')

In [ ]:
# Imports for the rest of the notebook.
import glob
import time
import copy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
)
import timm

from utils.env import data_dir, checkpoints_dir
from utils.plotting import (
    plot_curves, plot_confusion_matrix, show_image_grid, plot_roc_pr,
)
from utils.metrics import classification_report_dict
from utils.training import evaluate, fit
from utils.gradcam import GradCAM, overlay_cam, find_efficientnet_target_layer

## 4. Directory layout

We use torchvision's `ImageFolder` convention. Put your images here:

```
data/script_classifier/
  ltr/
    img_0001.jpg
    img_0002.jpg
    ...
  rtl/
    img_0001.jpg
    ...
```

`ImageFolder` will assign labels alphabetically, so `ltr -> 0`, `rtl -> 1` — exactly what we want.

If the main `data/` folder is empty (first run after clone), we fall back to
`sample_data/script_classifier/` which ships a few dozen tiny placeholder images so the notebook
can execute end-to-end immediately. Replace those with your real images for meaningful results.

In [ ]:
# Resolve the script-classifier dataset path, with a fall-back for fresh clones.
candidate_paths = [
    os.path.join(data_dir(), 'script_classifier'),
    os.path.join(repo_root, 'sample_data', 'script_classifier'),
]

dataset_root = None
for p in candidate_paths:
    if os.path.isdir(os.path.join(p, 'ltr')) and os.path.isdir(os.path.join(p, 'rtl')):
        dataset_root = p; break

if dataset_root is None:
    # Neither the real nor the sample data is present. Create a tiny synthetic scaffold so the
    # rest of the notebook at least runs: eight solid-colour dummy PNGs per class. Replace these
    # with real images before trusting any metric you get out!
    from PIL import Image
    dataset_root = os.path.join(data_dir(), 'script_classifier')
    for cls, colour in [('ltr', (230, 230, 230)), ('rtl', (160, 140, 110))]:
        d = os.path.join(dataset_root, cls); os.makedirs(d, exist_ok=True)
        for i in range(8):
            img = Image.new('RGB', (256, 256), colour)
            img.save(os.path.join(d, f'dummy_{i:02d}.png'))
    print('[warn] No real data found. Wrote 16 placeholder dummy images so the pipeline runs.')
    print('[warn] Replace the contents of', dataset_root, 'before trusting any metric.')

print('dataset_root =', dataset_root)
for cls in ['ltr', 'rtl']:
    n = len(glob.glob(os.path.join(dataset_root, cls, '*')))
    print(f'  {cls}: {n} images')

## 5. Load with `ImageFolder` and inspect the class balance

We load once with an identity transform so we can eyeball a preview grid before we commit to
augmentation. We also count per-class samples; if the imbalance ratio is >2×, we'll add class
weights or a weighted sampler (cf. notebook 09).

In [ ]:
identity = transforms.Lambda(lambda x: x)
ds_raw = datasets.ImageFolder(root=dataset_root, transform=identity)
CLASS_NAMES = ds_raw.classes   # ['ltr', 'rtl']
print('classes:', CLASS_NAMES)
print('total images:', len(ds_raw))

labels = [lbl for _, lbl in ds_raw.samples]
counts = Counter(labels)
for k in sorted(counts):
    print(f'  {CLASS_NAMES[k]} ({k}): {counts[k]}')

ratio = max(counts.values()) / max(1, min(counts.values()))
print(f'imbalance ratio: {ratio:.2f}x')
use_class_weights = ratio > 2.0
print('use class weights:', use_class_weights)

## 6. Preview samples

Always, always look at the data. We show 8 LTR and 8 RTL samples. Red flags to watch for:

- All LTR on white backgrounds and all RTL on coloured ones → model will cheat.
- All RTL photographed from extreme angles and all LTR flat → model learns perspective, not script.
- Any mislabelled sample (e.g., a pure English image accidentally in `rtl/`).

In [ ]:
rng = np.random.RandomState(0)
ltr_idx = [i for i, l in enumerate(labels) if l == 0]
rtl_idx = [i for i, l in enumerate(labels) if l == 1]
pick_ltr = rng.choice(ltr_idx, size=min(8, len(ltr_idx)), replace=False)
pick_rtl = rng.choice(rtl_idx, size=min(8, len(rtl_idx)), replace=False)
imgs, names = [], []
for i in list(pick_ltr) + list(pick_rtl):
    img, lbl = ds_raw[int(i)]
    imgs.append(img); names.append(CLASS_NAMES[lbl])
show_image_grid(imgs, labels=names, ncols=4, figsize=(12, 8))

## 7. Augmentation pipeline — critical caveat on horizontal flipping

> ### ⚠️ **CALLOUT: Horizontal flip is DISABLED for this task.**
>
> This is the single most important domain-specific decision in the notebook.
>
> **Why?** A horizontal flip mirrors the image left-to-right. When you flip an English image, the
> letters appear reversed — visually it looks a bit like right-to-left text to a CNN that is
> learning low-level stroke statistics. Conversely, flipping an Arabic image makes it look like
> garbled left-to-right text.
>
> **The consequence is brutal**: every horizontally-flipped sample is effectively **mislabelled**
> from the model's perspective. Training with HFlip actively teaches the model that both classes
> look the same. Val accuracy will cap somewhere near 50% and you'll stare at training curves
> for an hour wondering what broke.
>
> **Rule of thumb:** when your label depends on an asymmetric visual feature (reading direction,
> digit parity — is this a '6' or a '9'? — left-vs-right arrows, etc.), **always disable
> horizontal flip**.
>
> Vertical flip is *also* out for the same reason.

With that constraint respected, we still want strong augmentation: ColorJitter, RandomPerspective,
RandomErasing, and RandomResizedCrop. None of those invert left-right, so they're all safe.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    # NO RandomHorizontalFlip. NO RandomVerticalFlip. (See callout above.)
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 8. Stratified 70/15/15 split

We split by index (stratified by label) and wrap each split with its own transform via a small
dataset wrapper. Same pattern as notebook 12.

In [ ]:
class ScriptSplit(torch.utils.data.Dataset):
    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = list(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, lbl = self.base[self.indices[i]]
        return self.transform(img), int(lbl)

all_idx = np.arange(len(ds_raw))
all_lbl = np.asarray(labels)

# With very small datasets (the placeholder case), stratify may still work for 2 classes
# as long as each class has >=2 samples.
idx_trainval, idx_test, y_trainval, y_test = train_test_split(
    all_idx, all_lbl, test_size=0.15, stratify=all_lbl, random_state=42,
)
idx_train, idx_val, y_train, y_val = train_test_split(
    idx_trainval, y_trainval, test_size=0.15/0.85, stratify=y_trainval, random_state=42,
)
train_ds = ScriptSplit(ds_raw, idx_train, train_tfm)
val_ds   = ScriptSplit(ds_raw, idx_val,   eval_tfm)
test_ds  = ScriptSplit(ds_raw, idx_test,  eval_tfm)
print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

## 9. DataLoaders — with optional `WeightedRandomSampler`

If the class-balance check above flagged an imbalance >2×, we use `WeightedRandomSampler` so each
training batch is roughly balanced. Otherwise plain shuffling is fine.

In [ ]:
BATCH = 32
NUM_WORKERS = 2 if IN_COLAB else 0

if use_class_weights:
    # weight per sample = 1 / count(its class)
    cls_counts_tr = Counter(y_train.tolist())
    sample_weights = [1.0 / cls_counts_tr[int(y)] for y in y_train]
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True,
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True)
    print('using WeightedRandomSampler for balanced batches')
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    print('using plain shuffling (classes are balanced enough)')

val_loader  = DataLoader(val_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

## 10. Model — EfficientNetV2-S with a 2-class head

We use `num_classes=2` rather than `num_classes=1` with BCE loss. The two-logit setup keeps the
code identical to all previous notebooks (same CrossEntropy, same metrics), and it gives us a
calibrated softmax probability per class for threshold tuning later.

In [ ]:
model = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k', pretrained=True, num_classes=2,
)
model.to(device)
print('head:', model.get_classifier())
print('params:', sum(p.numel() for p in model.parameters())/1e6, 'M')

## 11. Stage 1 — Head-only warm-up (3 epochs)

Freezing the backbone and training only the classifier head for a few epochs is the fastest way
to get a decent starting point. The head starts from random weights; if we unfreeze everything
immediately, the first few batches' huge head gradients back-propagate into the backbone and
smash the pretrained features we're trying to preserve.

**Why only 3 epochs?** Beyond that the head has essentially converged against the frozen features.
More head-only training doesn't help; we want to unfreeze and let the backbone adapt.

In [ ]:
# Freeze everything except the classifier head.
for p in model.parameters():
    p.requires_grad_(False)
classifier = model.get_classifier()
for p in classifier.parameters():
    p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'stage-1 trainable params: {trainable:.3f}M')

# Class weights go into CE if we flagged imbalance earlier.
if use_class_weights:
    counts_full = Counter(labels)
    w = torch.tensor([1.0/counts_full[c] for c in range(2)], dtype=torch.float32)
    w = (w / w.sum() * 2).to(device)  # normalise so the mean weight is 1
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    print('class weights:', w.cpu().numpy())
else:
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=1e-3, weight_decay=1e-4)

stage1_ckpt = os.path.join(checkpoints_dir(), 'nb13_stage1.pt')
history1 = fit(
    model, train_loader, val_loader, loss_fn, opt1,
    epochs=3, device=device,
    checkpoint_path=stage1_ckpt, verbose=True,
)
print('stage 1 best val acc:', history1['best_val_acc'])

## 12. Stage 2 — Unfreeze + discriminative LR (10 epochs)

Unfreeze everything. The backbone now uses a **tiny** LR (1e-5) so we nudge the features gently,
and the head uses a smaller LR than stage 1 (1e-4) because it's already calibrated. Pair that
with cosine annealing for a smooth decay toward the end of training.

In [ ]:
# Unfreeze all parameters.
for p in model.parameters():
    p.requires_grad_(True)

head_param_ids = {id(p) for p in model.get_classifier().parameters()}
backbone_params, head_params = [], []
for p in model.parameters():
    (head_params if id(p) in head_param_ids else backbone_params).append(p)

opt2 = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': head_params,     'lr': 1e-4, 'weight_decay': 1e-4},
])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=10)

stage2_ckpt = os.path.join(checkpoints_dir(), 'nb13_script_classifier.pt')
history2 = fit(
    model, train_loader, val_loader, loss_fn, opt2,
    epochs=10, device=device,
    scheduler=sched2,
    early_stopping_patience=4,
    checkpoint_path=stage2_ckpt, verbose=True,
)
print('stage 2 best val acc:', history2['best_val_acc'])

In [ ]:
# Plot stage-2 curves.
plot_curves(history2, title='Script classifier — stage 2 (fine-tune)')

## 13. Load the best stage-2 checkpoint and evaluate on the test set

Binary classification evaluation is richer than it first looks. We'll compute:

- Precision, recall, F1 per class
- 2×2 confusion matrix
- ROC curve + AUC
- PR curve + average precision
- A threshold tuned for a specific recall target on the minority/critical class

In [ ]:
ckpt = torch.load(stage2_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

all_logits, all_y = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        all_logits.append(model(xb.to(device)).cpu().numpy())
        all_y.append(yb.numpy())
test_logits = np.concatenate(all_logits); test_y = np.concatenate(all_y)
test_prob = torch.softmax(torch.from_numpy(test_logits), dim=1).numpy()
test_pred = test_logits.argmax(axis=1)
acc = (test_pred == test_y).mean()
print(f'test accuracy: {acc:.4f}')
print()
print(classification_report(test_y, test_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
# 2x2 confusion matrix.
plot_confusion_matrix(test_y, test_pred, class_names=CLASS_NAMES, normalize=False,
                      figsize=(4.5, 4.5), title='Script classifier — test CM')

## 14. ROC and PR curves

For a binary problem we have a single ROC curve. We treat **class 1 (RTL)** as the positive class —
in a downstream pipeline the cost of missing an RTL image (routing it to an LTR-only OCR) is
typically higher than the reverse, so we want to see recall on class 1 clearly.

In [ ]:
pos_prob = test_prob[:, 1]   # prob of RTL

fpr, tpr, roc_thr = roc_curve(test_y, pos_prob)
roc_auc = auc(fpr, tpr)
prec, rec, pr_thr = precision_recall_curve(test_y, pos_prob)
ap = average_precision_score(test_y, pos_prob)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC'); axes[0].legend()
axes[1].plot(rec, prec, label=f'AP={ap:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR'); axes[1].legend()
plt.tight_layout(); plt.show()

## 15. Threshold tuning for a product-level recall target

The default decision rule is `argmax`, i.e., threshold = 0.5. But in production you usually have
an **asymmetric** cost. Let's say the product requirement is:

> *"We must correctly route at least **98% of RTL images** to the Arabic OCR pipeline. False RTL
> classifications on LTR images are annoying but not catastrophic."*

We find the smallest threshold on `P(class=RTL)` that still achieves >=98% recall on class 1, and
report the corresponding precision. This is a standard operating-point trade-off.

In [ ]:
TARGET_RECALL = 0.98

# Scan thresholds on the PR curve. sklearn's `precision_recall_curve` returns one fewer threshold
# than precision/recall points, so we only consider indices where pr_thr is defined.
valid = slice(0, len(pr_thr))
prec_v, rec_v, thr_v = prec[valid], rec[valid], pr_thr

eligible = np.where(rec_v >= TARGET_RECALL)[0]
if len(eligible) == 0:
    print(f'[warn] No threshold achieves recall >= {TARGET_RECALL}. Using argmax of recall.')
    best_idx = int(np.argmax(rec_v))
else:
    # pick the threshold with the highest precision among those meeting the recall floor
    best_idx = eligible[int(np.argmax(prec_v[eligible]))]

best_thr = float(thr_v[best_idx])
best_prec = float(prec_v[best_idx])
best_rec  = float(rec_v[best_idx])
print(f'chosen threshold on P(RTL): {best_thr:.4f}')
print(f'  -> precision on RTL: {best_prec:.4f}')
print(f'  -> recall    on RTL: {best_rec:.4f}')

# Re-compute predictions using the tuned threshold.
tuned_pred = (pos_prob >= best_thr).astype(int)
print()
print(classification_report(test_y, tuned_pred, target_names=CLASS_NAMES, digits=4))

## 16. Grad-CAM — a sanity check, not just a pretty picture

Grad-CAM here is a **debugging tool**, not a flashy demo. For this task we have a very specific
expectation:

> **The class-discriminative heatmap should be concentrated on the TEXT regions of the image.**
>
> If the heatmap is lighting up the background — a specific wall colour, sky, table texture,
> neon glow — then the model has learned a **spurious correlation** in your data. That means
> the training set has a hidden shortcut (e.g., all your Arabic images were shot in one
> particular café), and the model will fail in production on novel backgrounds.

If you see that pattern, the fix is data-collection, not modelling: shoot Arabic text on the
same backgrounds as English text, and vice versa.

In [ ]:
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)

correct = np.where(test_pred == test_y)[0]
wrong   = np.where(test_pred != test_y)[0]
pick = list(correct[:3]) + list(wrong[:3])
if len(pick) < 6:
    # top up with more correct samples if there are few errors
    extra = list(correct[3:3 + (6 - len(pick))])
    pick = pick + extra

target_layer = find_efficientnet_target_layer(model)
cam = GradCAM(model, target_layer)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, idx in zip(axes.ravel(), pick):
    xb, yb = test_ds[idx]
    cls = int(test_pred[idx])
    heatmap = cam(xb.unsqueeze(0).to(device), cls)
    img_unnorm = (xb * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    overlay = overlay_cam(img_unnorm, heatmap)
    colour = 'green' if cls == yb else 'red'
    ax.imshow(overlay); ax.axis('off')
    ax.set_title(f'pred: {CLASS_NAMES[cls]} | true: {CLASS_NAMES[int(yb)]}',
                 color=colour, fontsize=10)
plt.suptitle('Grad-CAM — does attention land on text, not background?', y=1.02)
plt.tight_layout(); plt.show()

## 17. ONNX export + numerical sanity check

ONNX is the lingua franca for model deployment. Once exported, the same graph runs on
`onnxruntime` (CPU, GPU, DirectML), `onnxruntime-mobile` (Android, iOS), TensorRT, OpenVINO, etc.

We export a fixed 224×224 RGB input and then verify that ONNX Runtime produces outputs that match
PyTorch to within ~1e-4. If they diverge, something went wrong during export (custom op,
non-standard activation).

In [ ]:
onnx_path = os.path.join(checkpoints_dir(), 'nb13_script_classifier.onnx')
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
print('exported ->', onnx_path, '(', round(os.path.getsize(onnx_path)/1e6, 1), 'MB)')

# Parity check with onnxruntime.
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    x_np = dummy.cpu().numpy()
    ort_out = sess.run(None, {'input': x_np})[0]
    with torch.no_grad():
        pt_out = model(dummy).cpu().numpy()
    diff = np.max(np.abs(ort_out - pt_out))
    print(f'max |onnx - pytorch| = {diff:.2e}  (<1e-4 is fine)')
except ImportError:
    print('[note] onnxruntime not installed; skipping parity check. pip install onnxruntime')

## 18. One-shot inference function

A thin wrapper so anyone downstream can call `predict_script('path/to/image.jpg')` and get back a
structured result. This is the function that would live in your service code.

In [ ]:
from PIL import Image

INFERENCE_TFM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def predict_script(image_path: str, threshold: float = best_thr) -> dict:
    """Classify an image as LTR vs RTL.

    Returns a dict: {'label': 'LTR'|'RTL', 'confidence': float, 'prob_rtl': float}.
    `threshold` is the tuned operating point on P(RTL) from section 15.
    """
    img = Image.open(image_path).convert('RGB')
    x = INFERENCE_TFM(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    prob_rtl = float(probs[1])
    if prob_rtl >= threshold:
        return {'label': 'RTL', 'confidence': prob_rtl, 'prob_rtl': prob_rtl}
    else:
        return {'label': 'LTR', 'confidence': float(probs[0]), 'prob_rtl': prob_rtl}

# demo call on the first test sample
demo_idx = int(idx_test[0])
demo_path = ds_raw.samples[demo_idx][0]
print('demo file:', os.path.basename(demo_path))
print('result   :', predict_script(demo_path))

## 19. Honest alternatives — when NOT to use this classifier

| Approach | Best when | Avoid when |
|---|---|---|
| **This deep classifier** | You need low-latency, zero-OCR inference at scale; you can collect 1k+ labelled images. | You have <500 images per class (you'll overfit; fall back to Unicode heuristic). |
| **OCR + Unicode range check** (`pytesseract` + `has_arabic`) | High-accuracy requirement, latency not critical (~200-500 ms). No training data. | Real-time pipelines, mobile deployment, stylised fonts where OCR struggles. |
| **Text-density heuristic** (binarize, measure left- vs right-side density) | Zero-dependency guard-rail when the classifier confidence is low. | Images with text evenly spread (paragraphs, multi-column documents). |

**Recommended hybrid**: run the classifier first. If `confidence < 0.7`, fall back to OCR + Unicode
as a tie-breaker. This keeps the fast path fast and only pays the OCR cost on ambiguous examples.


## 20. Deployment hints

Concrete integration pointers — treat these as breadcrumbs, not a full recipe:

**Backend (FastAPI)**

- Load the ONNX model once at application startup (not per-request).
- Wrap `predict_script` in a `POST /classify` endpoint that accepts a multipart image upload.
- Use `onnxruntime` with `CUDAExecutionProvider` if you have a GPU, else `CPUExecutionProvider`.
- Benchmark tip: on a modest CPU (4 cores) you should see ~30–50 ms per image at 224×224.

**Mobile (ONNX Runtime Mobile)**

- The exported `nb13_script_classifier.onnx` is ~80 MB. Apply **dynamic int8 quantization** via
  `onnxruntime.quantization.quantize_dynamic` to shrink it to ~20 MB with <1 pp accuracy drop.
- Deploy via `onnxruntime-mobile` on Android / iOS.

**Edge (ONNX Runtime on Raspberry Pi / Jetson)**

- Prefer EfficientNetV2-B0 if inference latency matters more than accuracy.
- Or distil a MobileNetV3-Small student from this EfficientNetV2-S teacher.

**Monitoring in production**

- Log `P(RTL)` for every request. A histogram that drifts over time (fewer confident
  predictions) is the first symptom of distribution shift — maybe new fonts, new phone cameras,
  new lighting. Retrain before performance degrades.


## Key Takeaways

- **Domain knowledge beats clever architecture.** The single most important decision in this
  notebook — *disable horizontal flip* — is a 30-second change that would otherwise silently
  destroy the model. Always inventory your augmentations against your label semantics.
- **Data collection is the real work.** A binary classifier is only as good as the diversity of
  its training images. If your RTL set is all shot in one location, Grad-CAM will tell you.
- **Two-stage training is a cheap win.** Head-only warm-up then discriminative-LR fine-tune is
  the default recipe for any transfer-learning problem on a small dataset.
- **Threshold tuning is not optional.** Argmax is rarely the right operating point for an
  asymmetric real-world cost. Tune for your product requirement.
- **Grad-CAM is a debugging tool.** If the heatmaps aren't on the text, your metrics are a lie.
- **ONNX gives you portability for free.** One export, many runtimes — CPU, GPU, mobile, edge.
- **Know when not to use ML.** OCR + Unicode check is a perfectly valid and sometimes *preferable*
  solution. Pick the tool that fits the constraint (latency, data, accuracy, cost).


## Exercises

1. **Break the model with HFlip.** Re-enable `RandomHorizontalFlip(p=0.5)` in `train_tfm`, retrain,
   and confirm val accuracy collapses toward 50%. Then fix it and feel the satisfaction.
2. **Three-class variant.** Add a third class — `mixed` — distinct from pure Arabic. How well can
   the model separate the three? (Hint: collect labelled mixed samples first.)
3. **Implement the text-density heuristic** as a pure-Python function and compare it head-to-head
   against the classifier on the test set. Where does each one fail?
4. **Hybrid fallback.** Implement the recommended hybrid: run the classifier; if
   `confidence < 0.7`, run an OCR + Unicode check. Measure the end-to-end accuracy vs each
   component alone.
5. **Quantize the ONNX model** with `onnxruntime.quantization.quantize_dynamic` to int8 and
   measure the accuracy drop (target: <1 pp) and inference speedup on CPU (typical: 2–3×).
6. **Cross-script domain shift.** Collect 20 images of **Urdu** or **Persian** text — both use
   the Arabic script — and evaluate the model on them without retraining. How does it do?
   Should it answer RTL? (Yes — and hopefully it does.)
7. **Distillation.** Train a MobileNetV3-Small student on this EfficientNetV2-S teacher. Compare
   model size, inference latency, and accuracy.
